In [3]:
# Install Hugging Face Transformers for using BERT.
!pip install -q transformers

# Install Hugging Face Datasets for loading AG News dataset.
!pip install -q datasets

# Install Accelerate to support training on GPU.
!pip install -q accelerate

# Install Gradio for creating a live web app.
!pip install -q gradio

# Install scikit-learn for accuracy and F1-score.
!pip install -q scikit-learn

In [4]:
# Import load_dataset function from Hugging Face Datasets.
from datasets import load_dataset

# Load AG News dataset using full Hugging Face repository name.
dataset = load_dataset("fancyzhx/ag_news")

# Print dataset structure to check train and test data.
print(dataset)

# Print one sample from training data.
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [5]:
# Import AutoTokenizer to load BERT tokenizer.
from transformers import AutoTokenizer

# Load tokenizer for bert-base-uncased model.
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Print confirmation message.
print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [6]:
# Define a function to tokenize news text.
def tokenize_function(example):

    # Convert text into BERT input format without fixed torch formatting.
    return tokenizer(
        example["text"],      # Take news text from dataset.
        truncation=True,      # Cut text if it is longer than allowed length.
        max_length=128        # Set maximum token length.
    )

# Apply tokenizer function to the dataset.
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Rename label column to labels because Trainer expects "labels".
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Remove original text column because model does not need raw text.
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Print confirmation message.
print("Dataset tokenized successfully.")

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Dataset tokenized successfully.


In [7]:
# Select 2000 training examples for faster Colab training.
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))

# Select 500 testing examples for faster evaluation.
test_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(500))

# Print training dataset size.
print("Training examples:", len(train_dataset))

# Print testing dataset size.
print("Testing examples:", len(test_dataset))

Training examples: 2000
Testing examples: 500


In [8]:
# Import BERT model class for sequence classification.
from transformers import AutoModelForSequenceClassification

# Load bert-base-uncased model for 4-class classification.
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",  # Use pre-trained BERT base uncased model.
    num_labels=4          # AG News has 4 classes.
)

# Print confirmation message.
print("BERT classification model loaded successfully.")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT classification model loaded successfully.


In [9]:
# Import numpy for prediction array handling.
import numpy as np

# Import accuracy_score and f1_score from scikit-learn.
from sklearn.metrics import accuracy_score, f1_score

# Define function to calculate accuracy and F1-score.
def compute_metrics(eval_pred):

    # Separate model raw outputs and true labels.
    logits, labels = eval_pred

    # Convert raw model outputs into predicted class numbers.
    predictions = np.argmax(logits, axis=-1)

    # Calculate accuracy score.
    accuracy = accuracy_score(labels, predictions)

    # Calculate weighted F1-score.
    f1 = f1_score(labels, predictions, average="weighted")

    # Return metrics as dictionary.
    return {
        "accuracy": accuracy,  # Return accuracy.
        "f1": f1               # Return F1-score.
    }

# Print confirmation message.
print("Metrics function created successfully.")

Metrics function created successfully.


In [10]:
# Import TrainingArguments to define training configuration.
from transformers import TrainingArguments

# Define training settings for BERT fine-tuning.
training_args = TrainingArguments(
    output_dir="./bert_news_model",        # Folder to save model outputs.
    eval_strategy="epoch",                # Evaluate model after each epoch.
    save_strategy="epoch",                # Save model after each epoch.
    learning_rate=2e-5,                   # Learning rate for fine-tuning.
    per_device_train_batch_size=8,        # Training batch size.
    per_device_eval_batch_size=8,         # Evaluation batch size.
    num_train_epochs=2,                   # Train model for 2 epochs.
    weight_decay=0.01,                    # Regularization to reduce overfitting.
    load_best_model_at_end=True,          # Load best model after training.
    metric_for_best_model="f1",           # Select best model using F1-score.
    report_to="none"                      # Disable external logging.
)

# Print confirmation message.
print("Training arguments created successfully.")

Training arguments created successfully.


In [11]:
# Import Trainer class from Hugging Face Transformers.
from transformers import Trainer

# Import DataCollatorWithPadding for dynamic padding during training.
from transformers import DataCollatorWithPadding

# Create data collator to convert batches into tensors safely.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create Trainer object for model training.
trainer = Trainer(
    model=model,                         # Pass BERT classification model.
    args=training_args,                  # Pass training settings.
    train_dataset=train_dataset,         # Pass training dataset.
    eval_dataset=test_dataset,           # Pass testing dataset.
    data_collator=data_collator,         # Convert batches into correct tensor format.
    compute_metrics=compute_metrics      # Pass accuracy and F1-score function.
)

# Start training the BERT model.
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.404790,0.878000,0.878190
2,0.416034,0.432070,0.886000,0.885503


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=500, training_loss=0.4160340576171875, metrics={'train_runtime': 89.8156, 'train_samples_per_second': 44.536, 'train_steps_per_second': 5.567, 'total_flos': 164903703991104.0, 'train_loss': 0.4160340576171875, 'epoch': 2.0})

In [12]:
# Evaluate trained model on test dataset.
results = trainer.evaluate()

# Print final evaluation results.
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.416034,0.432070,2,0.886000,0.885503


{'eval_loss': 0.432070255279541, 'eval_accuracy': 0.886, 'eval_f1': 0.8855031646949199}


In [13]:
# Save trained BERT model.
trainer.save_model("./saved_bert_news_model")

# Save tokenizer with the model.
tokenizer.save_pretrained("./saved_bert_news_model")

# Print confirmation message.
print("Model and tokenizer saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully.


In [14]:
# Import torch for prediction.
import torch

# Define AG News label names.
label_names = ["World", "Sports", "Business", "Sci/Tech"]

# Define prediction function.
def predict_news_topic(text):

    # Convert input headline into BERT tokens.
    inputs = tokenizer(
        text,                  # Input news headline.
        return_tensors="pt",   # Return PyTorch tensors.
        truncation=True,       # Cut long text if needed.
        padding=True,          # Add padding if needed.
        max_length=128         # Maximum token length.
    )

    # Move input tensors to same device as model.
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    # Disable gradient calculation during prediction.
    with torch.no_grad():

        # Get model output.
        outputs = model(**inputs)

    # Select class with highest score.
    predicted_class = torch.argmax(outputs.logits, dim=1).item()

    # Return predicted class name.
    return label_names[predicted_class]

# Create sample headline.
headline = "Apple launches new artificial intelligence features for iPhone users"

# Predict topic of sample headline.
prediction = predict_news_topic(headline)

# Print headline.
print("Headline:", headline)

# Print predicted topic.
print("Predicted Topic:", prediction)

Headline: Apple launches new artificial intelligence features for iPhone users
Predicted Topic: Sci/Tech


In [15]:
# Import Gradio for web app.
import gradio as gr

# Import torch for prediction.
import torch

# Define class labels.
label_names = ["World", "Sports", "Business", "Sci/Tech"]

# Define prediction function with probabilities.
def predict_with_probabilities(text):

    # Convert input text into BERT tokens.
    inputs = tokenizer(
        text,                  # User input headline.
        return_tensors="pt",   # Return PyTorch tensors.
        truncation=True,       # Cut long input if needed.
        padding=True,          # Add padding if needed.
        max_length=128         # Maximum token length.
    )

    # Move input tensors to model device.
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    # Disable gradient calculation.
    with torch.no_grad():

        # Get model output.
        outputs = model(**inputs)

    # Convert logits into probabilities.
    probabilities = torch.softmax(outputs.logits, dim=1)[0]

    # Create dictionary of class names and probabilities.
    result = {
        label_names[i]: float(probabilities[i])
        for i in range(len(label_names))
    }

    # Return result.
    return result

# Create Gradio interface.
interface = gr.Interface(
    fn=predict_with_probabilities,  # Prediction function.
    inputs=gr.Textbox(              # Text input box.
        lines=4,                    # Text box lines.
        placeholder="Enter a news headline here..."  # Placeholder text.
    ),
    outputs=gr.Label(num_top_classes=4),  # Show top 4 class probabilities.
    title="News Topic Classifier Using BERT",  # App title.
    description="This app classifies news headlines into World, Sports, Business, or Sci/Tech."  # App description.
)

# Launch app and create public link.
interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ebb17948592b6157c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
# Zip the saved model folder.
!zip -r saved_bert_news_model.zip saved_bert_news_model

  adding: saved_bert_news_model/ (stored 0%)
  adding: saved_bert_news_model/tokenizer.json (deflated 71%)
  adding: saved_bert_news_model/model.safetensors (deflated 7%)
  adding: saved_bert_news_model/config.json (deflated 55%)
  adding: saved_bert_news_model/tokenizer_config.json (deflated 43%)
  adding: saved_bert_news_model/training_args.bin (deflated 54%)


In [17]:
# Import files module from Google Colab.
from google.colab import files

# Download zipped model file.
files.download("saved_bert_news_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
# Evaluate trained model on test dataset.
results = trainer.evaluate()

# Print final results.
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.416034,0.432070,2,0.886000,0.885503


{'eval_loss': 0.432070255279541, 'eval_accuracy': 0.886, 'eval_f1': 0.8855031646949199}


In [19]:
# Save trained BERT model.
trainer.save_model("./saved_bert_news_model")

# Save tokenizer.
tokenizer.save_pretrained("./saved_bert_news_model")

# Print confirmation.
print("Model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.
